# Metacognitive Game Paper Reproducibility Analysis

This notebook reproduces the analyses and figures reported in the manuscript. Reusable analysis functions are defined in `paper_analysis_utils.py`, and all generated figures and tables are written to `paper_repro_outputs/`.

Analysis defaults in this notebook:
- Feature-distance models use main-task testing trials only, excluding training and tutorials.
- All 80 completers are the primary feature-distance sample.
- The strict above-chance sensitivity sample is `accuracy > .5` in all three test sessions (`N=51`).
- Questionnaire-task correlations are exploratory and include Bonferroni and Benjamini-Hochberg FDR corrections.

Reward normalization on this branch uses the maximum calibration error for the fixed confidence-response distribution. Rewarded score is a retrospective composite; see REWARD_NORMALIZATION_REVIEW.md for the before/after analysis and the separate historical-incentive question.


In [ ]:
import os

os.environ.setdefault("OUTDATED_IGNORE", "1")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

import paper_analysis_utils as u

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

paths = u.ensure_output_dirs("paper_repro_outputs")
paths

## 1. Load Data And Confirm Completers

The first two task CSVs contain participants who did not complete all three sessions. The corrected paper analyses use the 80 participants present in the final retest file as completers.

In [ ]:
data = u.load_paper_data("data")
session_table = u.session_completeness_table(data)
u.save_table(session_table, paths["tables"] / "session_completeness.csv")
session_table

## 2. Task Metric Summaries

Task metrics are computed from the testing block for each participant and session using the scoring functions in `analysisfunctions.py`.

In [ ]:
task_summaries = u.build_task_summaries(data)
u.save_table(task_summaries, paths["tables"] / "task_summaries.csv")

task_metric_means = task_summaries.groupby(["test", "test_label"])[u.TASK_METRICS].mean().reset_index()
u.save_table(task_metric_means, paths["tables"] / "task_metric_means_by_session.csv")
task_metric_means

## 3. Sample Definitions And Repeat-Sample Correction

The primary feature-distance analysis includes all participants who completed all three sessions (`N=80`). As a sensitivity analysis, we repeat the feature-distance analysis in participants whose accuracy exceeded chance-level performance (`accuracy > .5`) in all three test sessions (`N=51`).

The table below also reports several alternative threshold summaries so that the participant inclusion rule is explicit and reproducible.

In [ ]:
sample_definitions = u.sample_definition_table(task_summaries)
u.save_table(sample_definitions, paths["tables"] / "sample_definitions.csv")

strict_above_chance_ids = u.above_chance_subjects(task_summaries)
less_stringent_ids = u.less_stringent_performance_subjects(task_summaries)

print(f"Strict above-chance sensitivity sample: N={len(strict_above_chance_ids)}")
print(f"Less-stringent sensitivity sample: N={len(less_stringent_ids)}")
sample_definitions

## 4. Accuracy-Calibration Correlations

These correlations reproduce the manuscript's accuracy-calibration relationship in each session.

In [ ]:
accuracy_calibration = u.accuracy_calibration_correlations(task_summaries)
u.save_table(accuracy_calibration, paths["tables"] / "accuracy_calibration_correlations.csv")
fig = u.plot_accuracy_calibration(accuracy_calibration, task_summaries, paths["figures"] / "accuracy_calibration.svg")
plt.close(fig)
accuracy_calibration

## 5. Calibration Change Across Sessions

This section reproduces the repeated-measures ANOVA and the paired first-versus-final calibration comparison.

In [ ]:
calibration_means, calibration_anova, calibration_paired = u.calibration_change_tests(task_summaries)
u.save_table(calibration_means, paths["tables"] / "calibration_means.csv")
u.save_table(calibration_anova, paths["tables"] / "calibration_rm_anova.csv")
u.save_table(calibration_paired, paths["tables"] / "calibration_initial_vs_final_ttest.csv")

display(calibration_means)
display(calibration_anova)
display(calibration_paired)

## 6. Reliability of the Mean Across Three Administrations

Reliability is estimated for all 80 completers using ICC(2,k), with k=3: a two-way random-effects, absolute-agreement ICC for the mean across three administrations. This is not single-administration reliability. The existing 95% confidence intervals are exported as numeric bounds and displayed in the figure; negative lower bounds are retained.


In [ ]:
icc = u.icc_table(task_summaries)
u.save_table(icc, paths["tables"] / "icc2k_all_completers.csv")
fig = u.plot_icc_reliability(task_summaries, icc, paths["figures"] / "test_retest_reliability.svg")
plt.close(fig)
icc

## 7. Strategy Counts

Strategy summaries are computed from the 80 completers only, so initial and 1-week counts are not inflated by participants who did not complete the final session.

In [ ]:
strategy_tables = u.strategy_count_tables(data.completer_task_rows)
for name, table in strategy_tables.items():
    u.save_table(table, paths["tables"] / f"strategy_{name}.csv")

fig = u.plot_strategy_counts(strategy_tables, paths["figures"] / "strategy_counts.svg")
plt.close(fig)

display(strategy_tables["strategy_classification"])
display(strategy_tables["decision_process"])
display(strategy_tables["joint"])

## 8. Feature-Distance Analysis Using Testing Trials Only

Primary analysis: all 80 completers. Sensitivity analysis: strict above-chance subjects (`accuracy > .5` in all three test sessions, `N=51`).

Both analyses use main-task testing trials only (`use == "task"` and `block == "test"`), matching the Methods. Training and tutorial trials are excluded before distances and participant-by-distance accuracy means are calculated. There are 42 distinct testing trials per participant per session; the source CSV logs four rows per trial, so both source-row and distinct-trial counts are exported. Each session retains the same participant random-intercept model, `correct ~ distance`, fitted to participant-by-distance accuracy means.

The tables record trial inclusion, sample sizes, 95% slope intervals, and convergence. Previously saved pooled training/testing outputs are retained under `paper_repro_outputs/archive/` for provenance only. See `REVIEWER_REVISIONS.md` for the comparison and replacement Methods/Results wording.


In [ ]:
distance_models_all80, distance_grouped_all80 = u.fit_distance_models(data)
distance_models_strict51, distance_grouped_strict51 = u.fit_distance_models(data, strict_above_chance_ids)

u.save_table(distance_models_all80, paths["tables"] / "distance_models_all80_primary.csv")
u.save_table(distance_models_strict51, paths["tables"] / "distance_models_strict51_sensitivity.csv")
u.save_table(distance_grouped_all80, paths["tables"] / "distance_grouped_all80_primary.csv")
u.save_table(distance_grouped_strict51, paths["tables"] / "distance_grouped_strict51_sensitivity.csv")

fig = u.plot_distance_regression(distance_grouped_all80, paths["figures"] / "distance_regression_all80_primary.svg")
plt.close(fig)
fig = u.plot_distance_regression(distance_grouped_strict51, paths["figures"] / "distance_regression_strict51_sensitivity.svg")
plt.close(fig)

display(distance_models_all80)
display(distance_models_strict51)

## 9. Questionnaire Subscale Correlations

This reproduces the questionnaire subscale correlation matrix used to characterize questionnaire structure.

In [ ]:
questionnaire_covariance = u.questionnaire_subscale_covariance(data.questionnaire_subscales)
u.save_indexed_table(questionnaire_covariance, paths["tables"] / "questionnaire_subscale_correlations.csv")
fig = u.plot_questionnaire_covariance(questionnaire_covariance, paths["figures"] / "questionnaire_subscale_correlations.svg")
plt.close(fig)
questionnaire_covariance

## 10. Exploratory Questionnaire-Task Correlations With Multiple-Comparison Correction

Task metrics are averaged across sessions for each participant. The table includes uncorrected p-values, Bonferroni p-values, and Benjamini-Hochberg FDR p-values for the 66 all-subject mean subscale-by-task metric tests.

In [ ]:
questionnaire_task = u.questionnaire_task_correlations(data.questionnaire_subscales, task_summaries)
selected_questionnaire_task = u.selected_questionnaire_correlations(questionnaire_task)

u.save_table(questionnaire_task, paths["tables"] / "questionnaire_task_correlations_all80_corrected.csv")
u.save_table(selected_questionnaire_task, paths["tables"] / "questionnaire_task_selected_paper_correlations.csv")

fig = u.plot_questionnaire_task_heatmap(questionnaire_task, paths["figures"] / "questionnaire_task_correlation_heatmap.svg")
plt.close(fig)

print("Nominal p < .05:", int(questionnaire_task["significant_uncorrected"].sum()))
print("Bonferroni p < .05:", int(questionnaire_task["significant_bonferroni"].sum()))
print("BH-FDR p < .05:", int(questionnaire_task["significant_fdr_bh"].sum()))
selected_questionnaire_task

## 11. Strict-Sample Sensitivity For Selected Questionnaire Correlations

Questionnaire-task associations are exploratory. The main analysis uses all 80 completers; this sensitivity table shows the selected correlations after restricting to the strict `N=51` above-chance subgroup.

In [ ]:
questionnaire_task_strict51 = u.questionnaire_task_correlations(
    data.questionnaire_subscales, task_summaries, subject_ids=strict_above_chance_ids
)
selected_questionnaire_task_strict51 = u.selected_questionnaire_correlations(questionnaire_task_strict51)

u.save_table(questionnaire_task_strict51, paths["tables"] / "questionnaire_task_correlations_strict51_corrected.csv")
u.save_table(selected_questionnaire_task_strict51, paths["tables"] / "questionnaire_task_selected_strict51_sensitivity.csv")

selected_questionnaire_task_strict51

## 12. Power And Sample-Size Estimates

This section gives prospective planning estimates for Pearson correlations. It includes two plausible analysis families: the primary 66-test family using mean task metrics across sessions, and a larger 198-test family treating session-specific task metrics separately.

In [ ]:
power_66, current_power_66 = u.power_analysis_tables(n_tests=66)
power_198, current_power_198 = u.power_analysis_tables(n_tests=198)
power_66["family"] = "66 mean subscale-task tests"
power_198["family"] = "198 session-wise subscale-task tests"
current_power_66["family"] = "66 mean subscale-task tests"
current_power_198["family"] = "198 session-wise subscale-task tests"

power_required = pd.concat([power_66, power_198], ignore_index=True)
current_power = pd.concat([current_power_66, current_power_198], ignore_index=True)

u.save_table(power_required, paths["tables"] / "power_required_n.csv")
u.save_table(current_power, paths["tables"] / "power_current_n80.csv")

display(power_required)
display(current_power)

## 13. Corrected Paper Result Checkpoints

This final table collects the key corrected values to use when revising the manuscript.

In [ ]:
checkpoints = pd.DataFrame([
    {"result": "Completers", "value": int(task_summaries.subject_id.nunique())},
    {"result": "Strict above chance N", "value": int(len(strict_above_chance_ids))},
    {"result": "Accuracy-calibration initial r", "value": accuracy_calibration.loc[accuracy_calibration.test == 0, "r"].iloc[0]},
    {"result": "Accuracy-calibration 1-week r", "value": accuracy_calibration.loc[accuracy_calibration.test == 1, "r"].iloc[0]},
    {"result": "Accuracy-calibration final r", "value": accuracy_calibration.loc[accuracy_calibration.test == 2, "r"].iloc[0]},
    {"result": "Calibration ANOVA F", "value": calibration_anova.loc[calibration_anova.Source == "test", "F"].iloc[0]},
    {"result": "Calibration ANOVA GG p", "value": calibration_anova.loc[calibration_anova.Source == "test", "p-GG-corr"].iloc[0]},
    {"result": "Rewarded-score ICC2k", "value": icc.loc[icc.metric == "rewarded_score", "icc"].iloc[0]},
    {"result": "Questionnaire correlations surviving BH-FDR", "value": int(questionnaire_task["significant_fdr_bh"].sum())},
])
u.save_table(checkpoints, paths["tables"] / "corrected_paper_checkpoints.csv")
checkpoints

## 14. Supplementary Materials and Corrected Strategy Analysis

The strategy analysis accounts for repeated sessions using a participant random intercept. The primary model has additive categorical effects of classification strategy, decision strategy, and session; a separate sensitivity model includes all two-way interactions and tests those interactions. These specifications are chosen for identifiability and interpretation, not by selecting significant results. Strategy is self-reported, so associations do not establish causal effects.

The singleton `Wrote down` report is excluded from the strategy regressions only. Audit files show all 27 possible cells (including three empty cells) and matrix ranks: additive 7/7, two-way 19/19, full factorial 24/27. The full factorial model cannot identify all intended effects and is not fitted for inference. No three-way interaction claim is supported by this analysis.

Models use REML and asymptotic Wald chi-square tests. Unadjusted P values and BH-FDR-adjusted P values are exported, with separate 18-test families for the primary effects and sensitivity interactions. Coefficient estimates, 95% intervals, ranks, convergence, variance estimates, and fitting warnings are also saved. Statsmodels' absolute variance cutoff can emit a boundary warning for these small-scale outcomes; inspect the recorded variance ratios and covariance checks as well as convergence.

- **Supplemental Table 1**: corrected primary mixed-model tests, saved as CSV and a US-letter PDF.
- **Supplemental Figure 1**: descriptive accuracy and outcome variance by strategy and session, including the singleton report.
- **Supplemental Figures 2-6**: questionnaire subscale correlations by instrument.

The superseded OLS ANOVA CSV is retained under `paper_repro_outputs/archive/` solely to document the correction. Its P values must not be used for current conclusions.


In [ ]:
strategy_metrics = u.strategy_metric_table(data, task_summaries)
strategy_audit, strategy_cells = u.strategy_design_audit(strategy_metrics)
u.save_table(strategy_audit, paths["tables"] / "strategy_design_audit.csv")
u.save_table(strategy_cells, paths["tables"] / "strategy_cell_counts.csv")

supplemental_table_1, strategy_coefficients, strategy_diagnostics = u.strategy_metric_models(strategy_metrics)
strategy_interactions, interaction_coefficients, interaction_diagnostics = u.strategy_metric_models(
    strategy_metrics, model_kind="two_way"
)
u.save_table(supplemental_table_1, paths["tables"] / "supplemental_table_1_strategy_mixed_models.csv")
u.save_table(strategy_interactions, paths["tables"] / "strategy_two_way_interactions_sensitivity.csv")
u.save_table(pd.concat([strategy_coefficients, interaction_coefficients], ignore_index=True),
             paths["tables"] / "strategy_model_coefficients.csv")
u.save_table(pd.concat([strategy_diagnostics, interaction_diagnostics], ignore_index=True),
             paths["tables"] / "strategy_model_diagnostics.csv")

fig = u.save_supplemental_table_1_pdf(
    supplemental_table_1, paths["supplement"] / "supplemental_table_1.pdf",
    int(strategy_audit.iloc[0]["n_rows"]), int(strategy_audit.iloc[0]["n_subjects"]),
)
plt.close(fig)

fig = u.plot_supplemental_figure_1(strategy_metrics, paths["supplement"] / "supplemental_figure_1.pdf")
plt.close(fig)

supplemental_figure_paths = u.plot_subscale_correlation_figures(
    data.questionnaire_subscales, task_summaries, paths["supplement"]
)
print("Supplement pages written:")
for name in ["supplemental_table_1"] + ["supplemental_figure_1"] + sorted(supplemental_figure_paths):
    print(" ", paths["supplement"] / f"{name}.pdf")

display(strategy_audit)
display(supplemental_table_1)
display(strategy_interactions)


## Generated Outputs

All generated artifacts are written under `paper_repro_outputs/`:
- Figures: `paper_repro_outputs/figures/`
- Tables: `paper_repro_outputs/tables/`
- Supplement PDF pages (US letter): `paper_repro_outputs/supplement/`